In [ ]:
import os
import csv
import sqlite3

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("dark_background")

# Task 0
Data extraction: get the data from 3 tables & combine it into single `.csv` file.
After that read this file using pandas to create Dataframe.
So it will be all joined data in 1 dataframe. Quick check - should be 74818 rows in it.

In [ ]:
DB_PATH = "/db.sqlite3"
CSV_PATH = "/data/order_items.csv"

connection = sqlite3.connect(DB_PATH)
cursor = connection.cursor()

cursor.execute("""
    SELECT
        oi.id AS order_item_id,
        oi.quantity AS quantity,
        o.id AS order_id,
        o.datetime AS order_datetime,
        p.id AS product_id,
        p.name AS product_name,
        p.price AS product_price
    FROM restaurant_orderitem AS oi
    JOIN restaurant_order AS o ON oi.order_id = o.id
    JOIN restaurant_product AS p ON oi.product_id = p.id
""")

rows = cursor.fetchall()
columns = [description[0] for description in cursor.description]

cursor.close()
connection.close()

os.makedirs(os.path.dirname(CSV_PATH), exist_ok=True)
with open(CSV_PATH, "w", newline="", encoding="utf-8") as csv_file:
    writer = csv.writer(csv_file)
    writer.writerow(columns)
    writer.writerows(rows)

df = pd.read_csv(CSV_PATH, parse_dates=["order_datetime"])
print(df.shape)
df.head()

# Task 1
Get Top 10 most popular products in restaurant sold by Quantity.
Count how many times each product was sold and create a pie chart with percentage of popularity (by quantity) for top 10 of them.

Example:

![pie chart](../demo/pie.png)

In [ ]:
def autopct_format(values):
    def my_format(pct):
        total = sum(values)
        val = int(round(pct * total / 100.0))
        return f"{pct:.1f}% ({val})"
    return my_format


top10_quantity = (
    df.groupby(["product_id", "product_name"])["quantity"]
    .sum()
    .reset_index()
    .sort_values("quantity", ascending=False)
    .head(10)
)

fig, ax = plt.subplots(figsize=(8, 8))
ax.pie(
    top10_quantity["quantity"],
    labels=top10_quantity["product_name"],
    autopct=autopct_format(top10_quantity["quantity"]),
    startangle=90,
)
ax.set_title("Top 10 positions in menu by quantity")
plt.tight_layout()
plt.show()

# Task 2
Calculate `Item Price` (Product Price * Quantity) for each Order Item in dataframe.
And Make the same Top 10 pie chart, but this time by `Item Price`. So this chart should describe not the most popular products by quantity, but which products (top 10) make the most money for restaurant. It should be also with percentage.

In [ ]:
df["item_price"] = df["product_price"] * df["quantity"]

top10_revenue = (
    df.groupby(["product_id", "product_name"])["item_price"]
    .sum()
    .reset_index()
    .sort_values("item_price", ascending=False)
    .head(10)
)

fig, ax = plt.subplots(figsize=(8, 8))
ax.pie(
    top10_revenue["item_price"],
    labels=top10_revenue["product_name"],
    autopct=autopct_format(top10_revenue["item_price"]),
    startangle=90,
)
ax.set_title("Top 10 positions in menu by revenue")
plt.tight_layout()
plt.show()

# Task 3
Calculate `Order Hour` based on `Order Datetime`, which will tell about the specific our the order was created (from 0 to 23). Using `Order Hour` create a bar chart, which will tell the total restaurant income based on the hour order was created. So on x-axis - it will be values from 0 to 23 (hours), on y-axis - it will be the total sum of order prices, which were sold on that hour.

Example:

![bar chart](../demo/bar.png)

In [ ]:
df["order_hour"] = df["order_datetime"].dt.hour

income_by_hour = (
    df.groupby("order_hour")["item_price"]
    .sum()
    .reindex(range(24), fill_value=0)
)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(income_by_hour.index, income_by_hour.values)
ax.set_xticks(range(24))
ax.set_xlabel("Order Hour")
ax.set_ylabel("Income")
ax.set_title("Profit by Order Hour")
plt.tight_layout()
plt.show()

busiest_hour = income_by_hour.idxmax()
print(f"Most profitable hour: {busiest_hour}:00, income: {income_by_hour.max():.2f}")

# Task 4
Make similar bar chart, but right now with `Order Day Of The Week` (from Monday to Sunday), and also analyze total restaurant income by each day of the week.

In [ ]:
df["order_day_of_week"] = df["order_datetime"].dt.day_name()

days_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
income_by_day = (
    df.groupby("order_day_of_week")["item_price"]
    .sum()
    .reindex(days_order)
)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(income_by_day.index, income_by_day.values)
ax.set_xlabel("Order Day Of The Week")
ax.set_ylabel("Income")
ax.set_title("Profit by Day of the Week")
plt.tight_layout()
plt.show()

busiest_day = income_by_day.idxmax()
print(f"Most profitable day: {busiest_day}, income: {income_by_day.max():.2f}")
print(f"Average income per order: {df.groupby('order_id')['item_price'].sum().mean():.2f}")